# Phase 2: Model Training on Vertex AI

In production ML, training is separated from the development environment.
We write a standalone Python training script and submit it to Vertex AI
as a Custom Training Job. Vertex AI provisions a compute instance, runs
the script, saves the model artifact to GCS, and shuts down automatically.

This means training is reproducible, scalable, and decoupled from the notebook.

In [1]:
PROJECT_ID = "churn-mlops-498321"
BUCKET_NAME = "churn-mlops-bucket-caston"
REGION = "us-central1"

In [2]:
training_script = '''
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pickle
import subprocess

BUCKET_NAME = "churn-mlops-bucket-caston"

# Load data from GCS
print("Loading data...")
train_df = pd.read_csv(f"gs://{BUCKET_NAME}/data/train.csv")
test_df = pd.read_csv(f"gs://{BUCKET_NAME}/data/test.csv")

X_train = train_df.drop(columns=["Churn"])
y_train = train_df["Churn"]
X_test = test_df.drop(columns=["Churn"])
y_test = test_df["Churn"]

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# Train model
# class_weight='balanced' tells the model to pay extra attention to the
# minority class (churners) to compensate for the 73/27 imbalance
print("Training model...")
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
print("Evaluating model...")
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

# Save model to GCS
print("Saving model...")
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

subprocess.run([
    "gsutil", "cp", "model.pkl",
    f"gs://{BUCKET_NAME}/models/model.pkl"
])
print("Model saved to GCS successfully")
'''

# Write the script to a file
with open("train.py", "w") as f:
    f.write(training_script)

print("train.py written successfully")

train.py written successfully


In [3]:
# Print the training script to verify it looks correct
with open("train.py", "r") as f:
    print(f.read())


import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pickle
import subprocess

BUCKET_NAME = "churn-mlops-bucket-caston"

# Load data from GCS
print("Loading data...")
train_df = pd.read_csv(f"gs://{BUCKET_NAME}/data/train.csv")
test_df = pd.read_csv(f"gs://{BUCKET_NAME}/data/test.csv")

X_train = train_df.drop(columns=["Churn"])
y_train = train_df["Churn"]
X_test = test_df.drop(columns=["Churn"])
y_test = test_df["Churn"]

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# Train model
# class_weight='balanced' tells the model to pay extra attention to the
# minority class (churners) to compensate for the 73/27 imbalance
print("Training model...")
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
print("Evaluating model...")
y_pred = model.predict(X_test)
y_prob = mo

## Submit Vertex AI Custom Training Job
We package our training script and submit it to Vertex AI.
Vertex AI provisions a compute instance, runs the script, logs the
output, and shuts down automatically when done.

This is the core of MLOps — training is a reproducible, submittable
job rather than something you run manually in a notebook.

In [5]:
from google.cloud import aiplatform

PROJECT_ID = "churn-mlops-498321"
BUCKET_NAME = "churn-mlops-bucket-caston"
REGION = "us-central1"

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=f"gs://{BUCKET_NAME}"
)

job = aiplatform.CustomTrainingJob(
    display_name="churn-training-rf",
    script_path="train.py",
    container_uri="us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0:latest",
    requirements=["scikit-learn", "fsspec", "gcsfs", "pandas"]
)

print("Submitting training job...")

job.run(
    machine_type="n1-standard-4",
    replica_count=1,
    sync=True
)

print("Training job complete!")

Submitting training job...
Training script copied to:
gs://churn-mlops-bucket-caston/aiplatform-2026-06-10-23:00:43.684-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://churn-mlops-bucket-caston/aiplatform-custom-training-2026-06-10-23:00:43.795 
View Training:
https://console.cloud.google.com/agent-platform/locations/us-central1/training/2960035917886128128?project=434019816081
CustomTrainingJob projects/434019816081/locations/us-central1/trainingPipelines/2960035917886128128 current state:
PIPELINE_STATE_RUNNING
View backing custom job:
https://console.cloud.google.com/agent-platform/locations/us-central1/training/932117012581187584?project=434019816081
CustomTrainingJob projects/434019816081/locations/us-central1/trainingPipelines/2960035917886128128 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/434019816081/locations/us-central1/trainingPipelines/2960035917886128128 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/43401